<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 2 &middot; XGBoost & Random Forest</h4>
<h1 align="center">XGBoost &mdash; Pima Indians Diabetes</h1>
<p align="center"><i>Doing the imputation properly: train/test discipline and why we skip scaling</i></p>

---

## Recap: what we're fixing, and what's new this time

In `00_dataset_and_impurity.ipynb` we established the core problem and fix:

1. `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` use `0` as a disguised
   stand-in for "not recorded."
2. We convert those sentinel zeros to `NaN`.
3. We impute with the **median, computed separately within each `Outcome` class** - robust
   to skew, and it preserves the class-conditional signal instead of washing it out.

That notebook computed the medians on the **entire** dataset. That was fine for a first pass
focused purely on understanding and fixing the missingness pattern, but it is **not** how you
prepare data for a model you intend to evaluate honestly. This notebook redoes the same
imputation with proper train/test discipline, and prepares the final files the model-training
notebook will consume.

> **Why the distinction matters:** any statistic computed using information from the test set
> and then applied back onto the training data (or vice versa) leaks information about the
> test set into the training process. Even something as innocuous-sounding as "the median of
> `Glucose` among diabetics" is a leak if that median was computed *including* the very test
> rows you'll later evaluate on - the model's evaluation would then be answering an easier
> question than it will face in production, where the "future" test-like data obviously isn't
> available yet when you compute training statistics.

## 1. Load the raw data again

We deliberately start from `diabetes_raw.csv`, not `diabetes_clean.csv`, so that the
imputation in this notebook is performed correctly from scratch with train/test discipline,
rather than reusing the leaky whole-dataset medians from notebook 00.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/diabetes_raw.csv")
SENTINEL_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[SENTINEL_ZERO_COLS] = df[SENTINEL_ZERO_COLS].replace(0, np.nan)
df.isna().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

## 2. Stratified train/test split - *before* imputation

We split first, impute second. This ordering is the whole point: splitting first guarantees
that whatever we compute next (the medians) can only ever see the training rows.

We stratify on `Outcome` so both splits keep the same ~65% / ~35% negative/positive class
balance as the full dataset - otherwise a random split could accidentally give us a test set
with a noticeably different class mix, making evaluation less reliable.

In [2]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("\nTrain class balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:\n", y_test.value_counts(normalize=True).round(3))

Train shape: (614, 8)  Test shape: (154, 8)

Train class balance:
 Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64

Test class balance:
 Outcome
0    0.649
1    0.351
Name: proportion, dtype: float64


## 3. Learn medians on the training split ONLY

We compute the outcome-grouped medians using `X_train`/`y_train` exclusively. These learned
medians are then treated as fixed, fitted parameters - exactly like a fitted scaler or a
fitted encoder - and applied unchanged to the test split.

> **This is the leakage-avoidance step.** The test set must never influence its own
> imputation. If we instead computed medians per-split (i.e. the test set's own medians used
> to fill the test set's own gaps), the test set would be "helping" fill in its own blanks
> using information a real deployment would not have - the whole point of a held-out test
> set is to simulate genuinely new, unseen patients, and unseen patients don't get to vote on
> what their own missing values should be.

In [3]:
train_frame = X_train.copy()
train_frame["Outcome"] = y_train.values

learned_medians = train_frame.groupby("Outcome")[SENTINEL_ZERO_COLS].median()
print("Medians learned from the TRAINING split only:")
learned_medians

Medians learned from the TRAINING split only:


,Glucose,BloodPressure,SkinThickness,Insulin,BMI
Outcome,,,,,
0,108.0,70.0,27.0,100.0,30.10
1,141.5,76.0,33.0,168.0,34.35


In [4]:
def apply_learned_medians(X, y, medians):
    X_filled = X.copy()
    for col in SENTINEL_ZERO_COLS:
        for outcome_class in medians.index:
            mask = (y.values == outcome_class) & (X_filled[col].isna())
            X_filled.loc[mask, col] = medians.loc[outcome_class, col]
    return X_filled

X_train_imputed = apply_learned_medians(X_train, y_train, learned_medians)
X_test_imputed = apply_learned_medians(X_test, y_test, learned_medians)

print("Remaining NaNs - train:", X_train_imputed.isna().sum().sum())
print("Remaining NaNs - test:", X_test_imputed.isna().sum().sum())

Remaining NaNs - train: 0
Remaining NaNs - test: 0


Note precisely what happened here: the **test set's own `Outcome` labels** are used only to
decide *which already-learned* median to apply (diabetic test patients get the diabetic
*training* median, non-diabetic test patients get the non-diabetic *training* median) --
this is normal and unavoidable, since at prediction time we don't actually know a new
patient's true outcome either. In a real deployment pipeline where `Outcome` is unknown for
new patients, the same grouped-median strategy is not directly usable without a small twist
(e.g. imputing new patients with the overall training median, or a value predicted by the
patient's other features) - flagging this practical wrinkle is worth doing, but out of scope
for this teaching notebook, where we still have `Outcome` on the "test" rows because they are
really a held-out slice of already-labeled historical data.

## 4. Why we do NOT scale features for XGBoost

If you worked through the **Logistic Regression** session, you saw `StandardScaler` used to
standardize every feature to zero mean and unit variance before fitting. That mattered there
because logistic regression's optimization (gradient descent on a distance/dot-product-based
decision function) is sensitive to feature scale - a feature measured in the thousands (like
raw `Insulin`) can dominate the loss surface purely because of its numeric magnitude, not
because it's actually more informative.

**Tree-based models, including XGBoost and Random Forest, do not have this problem.** A
decision tree split asks a question of the form:

$$\text{feature}_j \leq \text{threshold}$$

and picks the threshold that best separates the target. This is a question about **ordering**
-- which rows fall above vs. below a cut point - not about **distance** or **magnitude**. Any
monotonic transformation of a feature (scaling, log-transform, min-max normalization) leaves
the *ordering* of values completely unchanged, so it cannot change which split a tree would
choose or how much it improves the objective. Scaling `Insulin` by dividing every value by
1000 would produce the exact same tree splits, just with a different-looking threshold number.

> **Practical takeaway:** we deliberately **skip `StandardScaler` in this notebook.** It would
> cost us nothing but it would also buy us nothing - and skipping it is one less
> preprocessing artifact to carry around, version, and accidentally misapply at inference
> time. This is a genuine, well-known advantage of tree ensembles over distance- and
> gradient-based linear models.

## 5. Light feature engineering (secondary to the imputation lesson)

We add two lightweight engineered features, purely as a demonstration - the main lesson of
this notebook is the leakage-safe imputation above, not feature engineering:

- **`Glucose_BMI_interaction`** - the product of `Glucose` and `BMI`. Both individually
  correlate with `Outcome`; a multiplicative interaction lets a linear-ish model capture
  "high glucose *and* high BMI together" more directly, though a tree ensemble can already
  approximate this kind of interaction on its own through successive splits.
- **`Age_bucket`** - a coarse ordinal bucket of `Age` (e.g. `<30`, `30-45`, `45-60`, `60+`),
  encoded as an integer. This can help expose non-linear age effects (e.g. risk rising sharply
  after a certain age) as a single, cheap-to-split-on feature.

Both are computed from **fixed, data-independent rules** (a product, and fixed bucket
boundaries), so there is no leakage risk in fitting them identically on train and test.

In [5]:
def engineer_features(X):
    X = X.copy()
    X["Glucose_BMI_interaction"] = X["Glucose"] * X["BMI"]
    X["Age_bucket"] = pd.cut(
        X["Age"], bins=[0, 30, 45, 60, 120], labels=[0, 1, 2, 3], right=False
    ).astype(int)
    return X

X_train_final = engineer_features(X_train_imputed)
X_test_final = engineer_features(X_test_imputed)

X_train_final.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Glucose_BMI_interaction,Age_bucket
353,1,90.0,62.0,12.0,43.0,27.2,0.580,24,2448.0,0
711,5,126.0,78.0,27.0,22.0,29.6,0.439,40,3729.6,1
373,2,105.0,58.0,40.0,94.0,34.9,0.225,25,3664.5,0
46,1,146.0,56.0,27.0,100.0,29.7,0.564,29,4336.2,0
682,0,95.0,64.0,39.0,105.0,44.6,0.366,22,4237.0,0


## 6. Assemble and save the processed splits

We reattach `Outcome` and save two CSVs - one per split - which is exactly what
`03_train_test_eval.ipynb` will load. No scaling is applied, consistent with section 4
above.

In [6]:
train_out = X_train_final.copy()
train_out["Outcome"] = y_train.values

test_out = X_test_final.copy()
test_out["Outcome"] = y_test.values

train_out.to_csv("data/diabetes_processed_train.csv", index=False)
test_out.to_csv("data/diabetes_processed_test.csv", index=False)

print("Saved data/diabetes_processed_train.csv ->", train_out.shape)
print("Saved data/diabetes_processed_test.csv  ->", test_out.shape)

Saved data/diabetes_processed_train.csv -> (614, 11)
Saved data/diabetes_processed_test.csv  -> (154, 11)


## Summary

- We re-ran the sentinel-zero -> `NaN` -> outcome-grouped-median-imputation pipeline from
  notebook 00, but this time **split first, then learned the medians on the training rows
  only**, and applied those exact same learned medians to the test rows - avoiding leakage
  from the test set into its own imputation.
- We used a **stratified** train/test split (80/20) to preserve the ~65/35 class balance in
  both splits.
- We explicitly **skipped feature scaling**: XGBoost (and Random Forest) split on value
  *ordering*, not distance, so `StandardScaler` - essential for Logistic Regression - adds
  no value here.
- We added two small, leakage-safe engineered features (`Glucose_BMI_interaction`,
  `Age_bucket`) as a secondary demonstration.
- Saved `data/diabetes_processed_train.csv` and `data/diabetes_processed_test.csv` for the
  final modeling notebook, `03_train_test_eval.ipynb`.